In [65]:
import os
import pytz
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType, FloatType
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.getenv('GCS_BUCKET_NAME')
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
CREDS_PATH = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

IST =  pytz.timezone('Asia/Kolkata')
today = datetime.now(IST).strftime('%Y-%m-%d')

INPUT_PATH = f'gs://{BUCKET_NAME}/raw/flights/{today}/'
OUTPUT_PATH = f'gs://{BUCKET_NAME}/processed/flights/{today}/'

print(INPUT_PATH)
print(OUTPUT_PATH)

gs://flight-pipeline-487919-flight-raw/raw/flights/2026-02-27/
gs://flight-pipeline-487919-flight-raw/processed/flights/2026-02-27/


In [36]:

SCHEMA = StructType((
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("time_position", FloatType(), True),
    StructField("last_contact", FloatType(), True),
    StructField("longitude", FloatType(), True),
    StructField("latitude", FloatType(), True),
    StructField("baro_altitude", FloatType(), True),
    StructField("on_ground", BooleanType(), True),
    StructField("velocity", FloatType(), True),
    StructField("true_track", FloatType(), True),
    StructField("vertical_rate", FloatType(), True),
    StructField("geo_altitude", FloatType(), True),
    StructField("squawk", StringType(), True),
    StructField("ingested_at", StringType(), True),
    StructField("category", StringType(), True)
))

In [37]:
spark = (
    SparkSession.builder
    .appName("FlightPipeline")
    .config(
        "spark.jars.packages",
        "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.22"  # updated version
    )
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", CREDS_PATH)
    .config("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.driver.userClassPathFirst", "true")
    .config("spark.executor.userClassPathFirst", "true")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print("Spark session created successfully!")

Spark version: 3.5.1
Spark session created successfully!


In [21]:
spark.stop()

# Read from GCS

In [38]:
try:
    df_json = spark.read.option("multiline", "true").json(INPUT_PATH)
    print(f"\nSuccessfully read from GCS!")
except Exception as e:
    print(f"\nUnable to read from GCS! Error occured: {e}")


Successfully read from GCS!


In [39]:
df_json.printSchema()
df_json.show()

root
 |-- states: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- time: long (nullable = true)

+--------------------+----------+
|              states|      time|
+--------------------+----------+
|[[801638, AXB1577...|1772142389|
+--------------------+----------+



In [45]:
df_exploded = df_json.select(
    F.explode(F.col("states")).alias("flight"),  # each flight is now one row
    F.col("time").alias("snapshot_time")          # keep the snapshot timestamp
)
print(f"Total flights after explode: {df_exploded.count()}")
df_exploded.show(3, truncate=True)

Total flights after explode: 173
+--------------------+-------------+
|              flight|snapshot_time|
+--------------------+-------------+
|[801638, AXB1577 ...|   1772142389|
|[801640, AIC2336 ...|   1772142389|
|[80162f, AIC2452 ...|   1772142389|
+--------------------+-------------+
only showing top 3 rows



In [52]:
df_mapped = df_exploded.select(
    F.col("snapshot_time"),
    F.col("flight").getItem(0).alias("icao24"),
    F.col("flight").getItem(1).alias("callsign"),
    F.col("flight").getItem(2).alias("origin_country"),
    F.col("flight").getItem(3).cast("double").alias("time_position"),
    F.col("flight").getItem(4).cast("double").alias("last_contact"),
    F.col("flight").getItem(5).cast("double").alias("longitude"),
    F.col("flight").getItem(6).cast("double").alias("latitude"),
    F.col("flight").getItem(7).cast("double").alias("baro_altitude"),
    F.col("flight").getItem(8).cast("boolean").alias("on_ground"),
    F.col("flight").getItem(9).cast("double").alias("velocity"),
    F.col("flight").getItem(10).cast("double").alias("true_track"),
    F.col("flight").getItem(11).cast("double").alias("vertical_rate"),
    F.col("flight").getItem(13).cast("double").alias("geo_altitude"),  # index 12 is sensors, we skip it
    F.col("flight").getItem(14).alias("squawk"),
    F.col("flight").getItem(16).cast("integer").alias("category")
)

print(f"Total rows: {df_mapped.count()}")
print("\nSchema after mapping:")
df_mapped.printSchema()

print("\nSample data:")
df_mapped.show(5, truncate=True)

Total rows: 173

Schema after mapping:
root
 |-- snapshot_time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- origin_country: string (nullable = true)
 |-- time_position: double (nullable = true)
 |-- last_contact: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- baro_altitude: double (nullable = true)
 |-- on_ground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- true_track: double (nullable = true)
 |-- vertical_rate: double (nullable = true)
 |-- geo_altitude: double (nullable = true)
 |-- squawk: string (nullable = true)
 |-- category: integer (nullable = true)


Sample data:
+-------------+------+--------+--------------+-------------+-------------+---------+--------+-------------+---------+--------+----------+-------------+------------+------+--------+
|snapshot_time|icao24|callsign|origin_country|time_position| last_contact|longitude|la

# Begin Transformations

In [53]:
df_filtered = df_mapped.filter(F.col('icao24').isNotNull() & F.col('callsign').isNotNull() 
                                & F.col('latitude').isNotNull() & F.col('longitude').isNotNull())

record_count_after = df_filtered.count()
record_count_before = df_mapped.count()
total_records_lost = record_count_before - record_count_after

print(f"\nTotal Records: {record_count_before}")
print(f"\nFiltered Records: {record_count_after}")
print(f"\nTotal Records Lost: {total_records_lost}")


Total Records: 173

Filtered Records: 173

Total Records Lost: 0


In [55]:
df_transformed = df_filtered.withColumn('callsign', F.upper(F.trim(F.col('callsign'))))\
    .withColumn('altitude_feet', F.round(F.col('baro_altitude') * 3.2804, 2))\
    .withColumn('geo_altitude_feet', F.round(F.col('geo_altitude') * 3.2804, 2))\
    .withColumn('speed_kmh', F.round(F.col('velocity') * 3.6, 2))\
    .withColumn('vertical_rate_ftpm', F.round(F.col('vertical_rate') * 196.85, 2))\
    .withColumnRenamed("true_track", "heading_degrees")\
    .withColumn('ingested_at', F.to_timestamp(F.current_timestamp(), 'Asia/Kolkata'))\
    .withColumn("flight_category",
                F.when(F.col("category") == 0,  "No Info")
                .when(F.col("category") == 1,  "No ADS-B Emitter")
                .when(F.col("category") == 2,  "Light Aircraft")
                .when(F.col("category") == 3,  "Small Aircraft")
                .when(F.col("category") == 4,  "Large Aircraft")
                .when(F.col("category") == 5,  "High Vortex Large Aircraft")
                .when(F.col("category") == 6,  "Heavy Aircraft")
                .when(F.col("category") == 7,  "High Performance Aircraft")
                .when(F.col("category") == 8,  "Rotorcraft")
                .when(F.col("category") == 9,  "Glider")
                .when(F.col("category") == 10, "lighter Than Air")
                .when(F.col("category") == 11, "Parachutist")
                .when(F.col("category") == 12, "Ultralight")
                .when(F.col("category") == 15, "UAV")
                .when(F.col("category") == 16, "Surface Vehicle - Emergency")
                .when(F.col("category") == 17, "Surface Vehicle - Service")
                .otherwise("Unknown")
            )\
    .withColumn("flight_phase",
        F.when(F.col("on_ground") == True,                     "On Ground")
         .when(F.col("baro_altitude") < 1000,                  "Takeoff / Landing")
         .when(F.col("baro_altitude").between(1000, 6000),     "Climbing / Descending")
         .otherwise(                                            "Cruising")
    )\
    .drop('baro_altitude', 'geo_altitude', 'velocity', 'vertical_rate', "category")
    



df_transformed.show(5, truncate = False)

+-------------+------+--------+--------------+-------------+-------------+---------+--------+---------+---------------+------+-------------+-----------------+---------+------------------+--------------------------+---------------+------------+
|snapshot_time|icao24|callsign|origin_country|time_position|last_contact |longitude|latitude|on_ground|heading_degrees|squawk|altitude_feet|geo_altitude_feet|speed_kmh|vertical_rate_ftpm|ingested_at               |flight_category|flight_phase|
+-------------+------+--------+--------------+-------------+-------------+---------+--------+---------+---------------+------+-------------+-----------------+---------+------------------+--------------------------+---------------+------------+
|1772142389   |801638|AXB1577 |India         |1.77214233E9 |1.77214238E9 |88.4486  |22.6622 |true     |186.23         |1000  |174.98       |NULL             |204.91   |-127.95           |2026-02-27 03:38:15.801685|No Info        |On Ground   |
|1772142389   |801640|AI

# Data quality check

In [64]:
print("=================== Data Quality Check ===================")

total = df_transformed.count()
print(f"\nTotal Transformed Count: {total}")


print(f"\n--------Total Null Counts--------")
df_nulls = df_transformed.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_transformed.columns])
df_nulls.show(5, truncate = False)

print(f"\n--------Flight Phase Distribution--------")
df_transformed.groupBy("flight_phase")\
    .count()\
    .orderBy(F.col("count").desc())\
    .show()

print(f"\n--------Flight Category Distribution--------")
df_transformed.groupBy("flight_category")\
    .count()\
    .orderBy(F.col("count").desc())\
    .show(truncate = False)

print(f"\n--------Top 10 Origin Countries--------")
df_transformed.groupBy("origin_country")\
    .count()\
    .orderBy(F.col("count").desc())\
    .limit(10)\
    .show(truncate = False)




=================== Data Quality Check ===================

Total Transformed Count: 173

--------Total Null Counts--------
+-------------+------+--------+--------------+-------------+------------+---------+--------+---------+---------------+------+-------------+-----------------+---------+------------------+-----------+---------------+------------+
|snapshot_time|icao24|callsign|origin_country|time_position|last_contact|longitude|latitude|on_ground|heading_degrees|squawk|altitude_feet|geo_altitude_feet|speed_kmh|vertical_rate_ftpm|ingested_at|flight_category|flight_phase|
+-------------+------+--------+--------------+-------------+------------+---------+--------+---------+---------------+------+-------------+-----------------+---------+------------------+-----------+---------------+------------+
|0            |0     |0       |0             |0            |0           |0        |0       |0        |0              |103   |12           |16               |0        |12                |0     

# Write to Ouput Path

In [67]:
print(f"\nWriting transformed data to: {OUTPUT_PATH}")

df_transformed.write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"Successfully written to: {OUTPUT_PATH}")
print(f"Total rows written: {df_transformed.count()}")


Writing transformed data to: gs://flight-pipeline-487919-flight-raw/processed/flights/2026-02-27/


Successfully written to: gs://flight-pipeline-487919-flight-raw/processed/flights/2026-02-27/
Total rows written: 173
